#Paquetes necesarios

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import cv2  
import math 

from ultralytics import YOLO

Detector de clases además de matrículas, también detecta hacia que dirección van y si se salen de la pantalla. Y aplica un blur a coches y personas para mantener el anonimato.

In [ ]:
import cv2
import re
import csv
from ultralytics import YOLO
from paddleocr import TextRecognition

ocr = TextRecognition(model_name="en_PP-OCRv5_mobile_rec")
modelo_matriculas = YOLO('C:\\Users\\mench\\runs\\detect\\train-4\\weights\\best.pt')
model = YOLO('yolo11n.pt') 

textos_por_id = {}      
intentos_ocr = {}       
UMBRAL_OK = 0.90
MAX_INTENTOS = 5
CADA_N_FRAMES = 3

contador_frames = 0
avisos = set()
vehiculos_guardados = set()
matriculas_guardadas = set()
textos_matriculas = set()
video_path = "C0142.mp4"
cap = cv2.VideoCapture(video_path)
posiciones_anteriores = {}

alto, ancho = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)), int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))

with open('archivos_matriculas.csv', mode='w', newline='', encoding='utf-8') as file:
    escritor = csv.writer(file)
    escritor.writerow(['Fotograma', 'tipo_objeto', 'confianza', 'identificador_tracking', 'x1', 'y1', 'x2', 'y2', 'matricula_en_su_caso', 'mx1', 'my1', 'mx2', 'my2', 'texto_matricula'])

    while(True):

        ret, frame = cap.read()

        if not ret:
            print("Fin del vídeo.")
            break

        contador_frames += 1

        # Resultados de detectar personas, bicicletas y coches. Además de matrículas con nuestro modelo entrenado
        results = model.track(frame, classes=[0, 1, 2, 3, 5], persist=True, verbose=False)
        results_matriculas = modelo_matriculas.track(frame, persist=True, verbose=False, tracker="tracker_matriculas.yaml")

        # Diccionario para almacenar el total de cada clase en el fotograma actual
        conteo_actual = {}

        for r in results:
            for box in r.boxes:
                # Comprobar que el objeto tiene un ID de tracking asignado
                if box.id is not None:
                    track_id = int(box.id[0])
                    x1, y1, x2, y2 = map(int, box.xyxy[0])
                    cls = int(box.cls[0])

                    ############### Parte opcional filtro BLUR ##############

                    #Comprobamos si es una persona o un coche y palicamos el filtro de blur
                    #cls = int(box.cls[0])
                    #if cls == 0 or cls == 2:

                    #   roi = frame[y1:y2, x1:x2]

                    #   roi_blur = cv2.GaussianBlur(roi, (99, 99), 30)

                    #   frame[y1:y2, x1:x2] = roi_blur

                    #########################################################3

                    centro_x_actual = (x1 + x2) / 2

                    # Comprobar si ya teníamos registrada una posición anterior para este ID
                    if track_id in posiciones_anteriores:
                        centro_x_anterior = posiciones_anteriores[track_id]

                        if centro_x_actual > centro_x_anterior:
                            direccion = "Derecha"
                        elif centro_x_actual < centro_x_anterior:
                            direccion = "Izquierda"
                        else:
                            direccion = "Quieto"
                    else:
                        direccion = "Nuevo"

                    # Guardar la posición actual como "anterior" para el siguiente fotograma
                    posiciones_anteriores[track_id] = centro_x_actual

                    # Condiciones para comprobar por donde sale el objeto de la pantalla
                    if x1 <= 5:
                        if track_id not in avisos:
                            print(f"El objeto ID {track_id} está saliendo por la IZQUIERDA de la pantalla.")
                            avisos.add(track_id)

                    #Si x2 toca el borde derecho (mayor o igual al ancho total del vídeo)
                    if x2 >= (ancho - 5):
                        if track_id not in avisos:
                            print(f"El objeto ID {track_id} está saliendo por la DERECHA de la pantalla.")
                            avisos.add(track_id)

                    confidence = round(float(box.conf[0]), 2)
                    cls = int(box.cls[0])
                    nombre_clase = model.names[cls]

                    conteo_actual[nombre_clase] = conteo_actual.get(nombre_clase, 0) + 1

                    nombre_objeto = model.names[cls]

                    if track_id not in vehiculos_guardados:

                        escritor.writerow([contador_frames, nombre_objeto, confidence, track_id, x1, y1, x2, y2, False, None, None, None, None, None])

                        vehiculos_guardados.add(track_id)

                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    cv2.putText(frame, f"{nombre_clase} {confidence}", (x1, y1 - 10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
                    cv2.putText(frame, f"ID:{track_id} {direccion}", (x1, y1 - 30),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

        # Procesar detecciones del modelo de matrículas
        for rm in results_matriculas:
            for box in rm.boxes:
                if box.id is None:# evita el crash cuando el tracker no asigna ID
                    continue
                track_id = int(box.id[0])
                mx1, my1, mx2, my2 = map(int, box.xyxy[0])

                m = 5
                x1, y1 = max(0, mx1 - m), max(0, my1 - m)
                x2, y2 = min(ancho, mx2 + m), min(alto, my2 + m)
                recorte_matricula = frame[y1:y2, x1:x2]

                texto, score = textos_por_id.get(track_id, ("", 0.0))
                n = intentos_ocr.get(track_id, 0)

                if score < UMBRAL_OK and n < MAX_INTENTOS and contador_frames % CADA_N_FRAMES == 0:
                    m = 5
                    recorte = frame[max(0, my1-m):min(alto, my2+m), max(0, mx1-m):min(ancho, mx2+m)]
                    if recorte.size > 0:
                        res = ocr.predict(recorte)[0]
                        # TextRecognition devuelve 'rec_text'/'rec_score'; el pipeline PaddleOCR devuelve 'rec_texts'/'rec_scores'
                        nuevo = re.sub(r'[^A-Z0-9]', '', res['rec_text'].upper())
                        nuevo_score = float(res['rec_score'])
                        if nuevo_score > score:
                            texto, score = nuevo, nuevo_score
                            textos_por_id[track_id] = (texto, score)
                    intentos_ocr[track_id] = n + 1

                conf_mat = round(float(box.conf[0]), 2)
                cls_mat = int(box.cls[0])
                nombre_matricula = modelo_matriculas.names[cls_mat]

                lectura_final = score >= UMBRAL_OK or n >= MAX_INTENTOS
                if lectura_final and track_id not in matriculas_guardadas and texto:

                    escritor.writerow([contador_frames, nombre_matricula, conf_mat, track_id, None, None, None, None, True, mx1, my1, mx2, my2, texto])

                    matriculas_guardadas.add(track_id)
                    textos_matriculas.add(texto)

                conteo_actual[nombre_matricula] = conteo_actual.get(nombre_matricula, 0) + 1

                cv2.rectangle(frame, (mx1, my1), (mx2, my2), (255, 0, 0), 2)
                cv2.putText(frame, f"{nombre_matricula} {conf_mat}", (mx1, my1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
                cv2.putText(frame, texto, (mx1, my1 - 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
                cv2.putText(frame, f"ID:{track_id}", (mx1, my1 - 50),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

        #Dibujar el panel de recuento en la esquina superior izquierda del fotograma
        y_pos = 30
        for clase, cantidad in conteo_actual.items():
            texto = f"{clase}: {cantidad}"
            cv2.putText(frame, texto, (20, y_pos),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
            y_pos += 30

        cv2.imshow("frame", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

Detector de matrículos con muchos falsos positivos.

In [ ]:
video_path = "C0142.mp4"
cap = cv2.VideoCapture(video_path)

while(True):
    ret, frame = cap.read()
        
    if not ret:
        print("Fin del vídeo.")
        break
        
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    _, img_th1 = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY) 

    contornos, hierarchy = cv2.findContours(
        img_th1, 
        cv2.RETR_TREE, 
        cv2.CHAIN_APPROX_SIMPLE
    )
    
    for c in contornos:
        area = cv2.contourArea(c)
        if area < 500 or area > 700:
            continue
        
        x, y, w, h = cv2.boundingRect(c)
        
        if w > h:
            aspect_ratio = w / float(h)
            
            if 2.0 <= aspect_ratio <= 5.0:
                cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 255), 2)
                cv2.putText(frame, "Candidato Matricula", (x, y - 10), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2)

    cv2.imshow("Matriculas", frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()